# Information

Example notebook to show how to use and apply a **hard buffer** around known occurences (**labels**).<br>
Functionality can be used to both:
- Extent the existing labels around the known occurences by a certain number of pixels
- Remove potential negative labels around the known occurences by a certain number of pixels

Both options work with the same function. Buffer shape can either be a **circle** or a **square**.<br>

For this **example**, we used different variable names for the labels:

- `labels_array`: Original labels raster
- `buffered_labels`: Labels raster with a **circle** buffer around the known occurences
- `removed_negative_labels`: Labels raster with a **cicle** buffer around the known occurences, with negative labels removed

To apply the function in a real case scenario, simply always use  the `labels_array` name for all steps.

# Set up

In [1]:
import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

import numpy as np
import pandas as pd
import rasterio
import gc

from tqdm import tqdm
from pathlib import Path

# Custom modules
from beak.utilities.io import save_raster

BASE_PATH = files("beak.data")
gc.enable()


**User** inputs

In [2]:
ROOT_PATH = BASE_PATH / "PROCESSED" / "national_102008_2240_us_cont"
PATH_LABELS = ROOT_PATH  / "labels" / "EPSG_102008_RES_2240_us_cont_POCO_HM9_PCUDEPPRO.tif"
PATH_BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_102008_RES_2240_US_CONT.tif"


# 1: **Create** file list
...

# 2: **Load** data

### Base Raster

In [3]:
# Base raster
base_raster = rasterio.open(PATH_BASE_RASTER)
base_raster_array = base_raster.read(1)
base_raster_array = np.where(base_raster_array == base_raster.nodata, np.nan, base_raster_array)

### Labels

#### Original labels raster

In [4]:
# Labels raster
labels_raster = rasterio.open(PATH_LABELS)
labels_array = labels_raster.read(1)
labels_array = np.where(labels_array == labels_raster.nodata, np.nan, labels_array)

# Check the count of positives and negatives
print(np.unique(labels_array, return_counts=True))

(array([ 0.,  1., nan]), array([1610563,     138, 1039520], dtype=int64))


#### Apply hard buffer

1. Example to extent the positive labels by buffering one pixel around a circular neighborhood
2. Example to remove negative labels by buffering 5 pixels around a circular neighborhood

Both versions are so called **"hard buffers"**. <br>
Shape of the buffer can be either "**circle**" or "**square**".

Create buffer around **positive** labels in order to **extent** them and get more training labels.<br>
If no `buffer_value` is provided, the pixels within the buffer-zone will be set to the `positive_label_value`.

In [5]:
from beak.utilities.preparation import create_hard_buffer_around_labels

# Create extended positive labels
buffered_labels = create_hard_buffer_around_labels(labels_array, 
                                                   radius=1, 
                                                   shape="circle", 
                                                   positive_label_value=1)

# Check the new count of positives
print(np.unique(buffered_labels, return_counts=True))

(array([ 0.,  1., nan]), array([1610027,     674, 1039520], dtype=int64))


Remove potential **negatives** by buffering 3 pixels around the **new** positive labels from above.<br>
The difference between this option and extending the positive labels is another `buffer_value` than the `positive_label_value`.<br>
Setting the buffered area to a value **different from 0 and 1** will remove the negative labels later on in the preprocessing.<br>

The `buffer_value` can be set to `np.nan`, since any rows containing `np.nan` will be removed before modeling, or another **number**.<br>
In both ways, they will not be considered in the workflow for modeling.

In [6]:
# Remove negative labels
removed_negatives = create_hard_buffer_around_labels(buffered_labels,
                                                     radius=3,
                                                     shape="circle",
                                                     positive_label_value=1,
                                                     buffer_value=np.nan)

# Check the new count of positives
print(np.unique(removed_negatives, return_counts=True))

(array([ 0.,  1., nan]), array([1604873,     674, 1044674], dtype=int64))


Save the rasters to check in **GIS**

In [7]:
save_raster(Path("example_labels_original.tif"), labels_array, metadata=labels_raster.meta)
save_raster(Path("example_labels_positives_extended.tif"), buffered_labels, metadata=labels_raster.meta)
save_raster(Path("example_labels_negatives_removed.tif"), removed_negatives, metadata=labels_raster.meta)

# 3: **Remove** outliers
Go on with the data preprocessing steps etc. and choose which labels you want to use.